# Stage 1 — Generate Synthetic Student Data

Create a 40-row student dataset with **intentional data-quality issues**
(missing values, out-of-range entries, inconsistent city names, duplicates)
so the later stages have something meaningful to find and fix.

In [ ]:
# pip install pandas numpy scikit-learn openpyxl  (if not already installed)

import os
import pandas as pd
import numpy as np

DATA_DIR = os.path.join("..", "data")
os.makedirs(DATA_DIR, exist_ok=True)

## Create base student records

In [ ]:
np.random.seed(42)
n = 40

data = {
    "student_id": range(1, n + 1),
    "name": [f"Student_{i}" for i in range(1, n + 1)],
    "class": np.random.choice(["6A", "6B", "7A", "7B"], n),
    "gender": np.random.choice(["M", "F"], n),
    "attendance_pct": np.random.randint(55, 100, n).astype(float),
    "marks_math": np.random.randint(30, 100, n).astype(float),
    "marks_science": np.random.randint(30, 100, n).astype(float),
    "marks_english": np.random.randint(30, 100, n).astype(float),
    "city": np.random.choice(
        ["Delhi", "delhi ", "New Delhi", "Mumbai", "mumbai", "Jaipur"], n),
    "admission_date": pd.date_range("2021-04-01", periods=n, freq="9D"),
}
df = pd.DataFrame(data)
df.head()

## Inject realistic data-quality problems

We deliberately introduce issues so the pipeline has something to detect:
- **Missing values** in `marks_math` and `attendance_pct`
- **Impossible values** (attendance > 100 %, negative marks)
- **Duplicate row**

In [ ]:
missing_idx = np.random.choice(df.index, 6, replace=False)
df.loc[missing_idx[:3], "marks_math"] = np.nan          # missing marks
df.loc[missing_idx[3:], "attendance_pct"] = np.nan       # missing attendance
df.loc[2, "attendance_pct"] = 141                        # impossible value (>100%)
df.loc[5, "marks_science"] = -10                         # impossible value (<0)
df = pd.concat([df, df.iloc[[7]]], ignore_index=True)    # duplicate row

print(f"Shape : {df.shape}")
df.head(10)

## Save to Excel

In [ ]:
output_file = os.path.join(DATA_DIR, "student_data.xlsx")
df.to_excel(output_file, index=False, engine="openpyxl")
print(f"Saved : {output_file}")